# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


In [1]:
import sys
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql.functions import to_timestamp, datediff, col, trim, upper
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# --- 0. Inicialización del contexto de Glue (compatible con Notebooks y Jobs) ---
if '--JOB_NAME' in sys.argv:
    args = getResolvedOptions(sys.argv, ['JOB_NAME'])
else:
    args = {'JOB_NAME': 'jupyter_interactive_job'}

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

path_bronze = "s3://proyecto-ny311/bronze/311-service-requests-from-2010-to-present.csv"


Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: bf61b150-acc4-4871-ae1e-ee99c9f8cf09
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session bf61b150-acc4-4871-ae1e-ee99c9f8cf09 to get into ready status...
Session bf61b150-acc4-4871-ae1e-ee99c9f8cf09 has been created.



In [2]:

# --- 1. Lectura del CSV desde Bronze con manejo robusto de comillas ---
print("1. Leyendo CSV desde Bronze...")
df = spark.read.csv(
    path_bronze,
    header=True,
    inferSchema=True,
    quote='"',   # Indica que los campos pueden estar rodeados por comillas dobles
    escape='"'   # Indica que las comillas dobles internas están escapadas con otra comilla doble
)
print("CSV leído exitosamente.")


1. Leyendo CSV desde Bronze...
CSV leído exitosamente.


In [3]:
# --- 2. Limpieza inicial: Duplicados y Fechas ---

# 2.1 Conteo y Eliminación de Duplicados
print("\n2.1 Contando y eliminando duplicados por 'Unique Key'...")
conteo_original = df.count()
df_clean = df.dropDuplicates(["Unique Key"])
conteo_limpio = df_clean.count()
duplicados = conteo_original - conteo_limpio
print(f"Total de registros originales: {conteo_original}")
print(f"Total de registros después de eliminar duplicados: {conteo_limpio}")
print(f"Cantidad de duplicados eliminados: {duplicados}")

# 2.2 Primera y Última Fecha del Dataset
columna_fecha = "Created Date" # Asumiendo que esta es la columna de fecha de inicio
print(f"\n2.2 Determinando el rango de fechas en '{columna_fecha}'...")
df_clean.select(
    F.min(columna_fecha).alias("Primera_Fecha"),
    F.max(columna_fecha).alias("Ultima_Fecha")
).show()



2.1 Contando y eliminando duplicados por 'Unique Key'...
Total de registros originales: 21960000
Total de registros después de eliminar duplicados: 209999
Cantidad de duplicados eliminados: 21750001

2.2 Determinando el rango de fechas en 'Created Date'...
+-------------------+-------------------+
|      Primera_Fecha|       Ultima_Fecha|
+-------------------+-------------------+
|2019-09-22 11:38:12|2019-12-01 02:04:01|
+-------------------+-------------------+


In [4]:
df_clean.show(5, truncate=False) # truncate=False para ver el contenido completo de las columnas


+----------+-------------------+-------------------+------+--------------------------------+--------------------------+--------------------------------+-------------+------------+------------------+------------------+----------------------+----------------------+----------------------+----------------------+------------+-------------+----------------------------+-------------+-----------+--------+----------------------------------------------------------------------------------------------------+------------------------------+----------------+----------+-------------+--------------------------+--------------------------+----------------------+------------------+-------------+------------+--------------------+---------------------------------------------------+-------------------+------------------------+---------+----------------------+-----------------+------------------+-------------------------------------------------------------------------------------------------------------------

In [5]:
# --- 3. Análisis y Gestión de Valores Nulos ---

# 3.1 Calcular el porcentaje de nulos para todas las columnas
print("\n3.1 Calculando valores nulos por columna (detalle completo):")
total_rows_after_duplicates = df_clean.count()
exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_clean.columns]
df_nulls_summary = df_clean.select(*exprs)
pdf_nulls = df_nulls_summary.toPandas().T
pdf_nulls.columns = ['Valores Nulos']
pdf_nulls['Porcentaje Nulos (%)'] = (pdf_nulls['Valores Nulos'] / total_rows_after_duplicates) * 100
pdf_nulls_sorted = pdf_nulls.sort_values(by='Porcentaje Nulos (%)', ascending=False)
print(pdf_nulls_sorted.to_string()) # Usar to_string() para ver todas las filas



3.1 Calculando valores nulos por columna (detalle completo):
                                Valores Nulos  Porcentaje Nulos (%)
Due Date                               209999            100.000000
Bridge Highway Name                    209983             99.992381
Vehicle Type                           209891             99.948571
Bridge Highway Segment                 209786             99.898571
Road Ramp                              209785             99.898095
Bridge Highway Direction               209784             99.897619
Taxi Company Borough                   209781             99.896190
Taxi Pick Up Location                  204227             97.251415
Facility Type                          186832             88.968043
Address Type                           114806             54.669784
Landmark                               104439             49.733094
Intersection Street 2                   82505             39.288282
Intersection Street 1                   82496         

In [6]:
# 3.2 Definir y Eliminar Columnas con Nulos Excesivos o Redundantes
print("\n3.2 Eliminando columnas con excesivos nulos o redundantes para predicción de accidentalidad...")
cols_to_drop = [
    "Due Date", "Bridge Highway Name", "Vehicle Type",
    "Taxi Company Borough", "Road Ramp", "Bridge Highway Segment",
    "Bridge Highway Direction", "Taxi Pick Up Location",
    "Facility Type",
    "Location",         # Redundante con latitude/longitude
    "Zip Codes",        # Redundante con Incident Zip (mantener Incident Zip)
    "Address Type",     # 54.67% nulos, demasiado para ser útil
    "Landmark",         # 49.73% nulos
    "Intersection Street 2", # 39.29% nulos
    "Intersection Street 1", # 39.28% nulos
    "Cross Street 2",   # 31.28% nulos
    "Cross Street 1"    # 31.19% nulos
]
df_drop = df_clean.drop(*cols_to_drop)
print(f"Columnas eliminadas: {cols_to_drop}")



3.2 Eliminando columnas con excesivos nulos o redundantes para predicción de accidentalidad...
Columnas eliminadas: ['Due Date', 'Bridge Highway Name', 'Vehicle Type', 'Taxi Company Borough', 'Road Ramp', 'Bridge Highway Segment', 'Bridge Highway Direction', 'Taxi Pick Up Location', 'Facility Type', 'Location', 'Zip Codes', 'Address Type', 'Landmark', 'Intersection Street 2', 'Intersection Street 1', 'Cross Street 2', 'Cross Street 1']


In [7]:
print("\n Primeros 5 registros del DataFrame después de limpieza:")
df_drop.show(5, truncate=False) # truncate=False para ver el contenido completo de las columnas



 Primeros 5 registros del DataFrame después de limpieza:
+----------+-------------------+-------------------+------+--------------------------------+--------------------------+--------------------------------+-------------+------------+------------------+------------------+-------------+-----------+----------------------------------------------------------------------------------------------------+------------------------------+----------------+----------+-------------+--------------------------+--------------------------+----------------------+------------------+-------------+-----------------+------------------+-------------------+------------------+----------------------+----------------+
|Unique Key|Created Date       |Closed Date        |Agency|Agency Name                     |Complaint Type            |Descriptor                      |Location Type|Incident Zip|Incident Address  |Street Name       |City         |Status     |Resolution Description                                 

In [8]:

# 3.3 Imputación de campos de Texto (Categorías)
print("\n3.3 Imputando valores nulos en columnas categóricas...")
text_impute_map = {
    "Descriptor": "NOT PROVIDED",
    "Location Type": "UNKNOWN",
    "Incident Address": "UNKNOWN_ADDRESS",
    "Street Name": "UNKNOWN_STREET",
    "City": "UNSPECIFIED_CITY",
    "City Council Districts": "UNKNOWN_COUNCIL_DISTRICT", # 5.77% nulos
    "Police Precincts": "UNKNOWN_PRECINCT",             # 5.52% nulos
    "Community Districts": "UNKNOWN_COMMUNITY_DISTRICT", # 5.52% nulos
    "Borough Boundaries": "UNKNOWN_BOROUGH_BOUNDARY",   # 5.52% nulos
    "BBL": "UNKNOWN_BBL",                               # 13.92% nulos, tratado como categórico
    "Resolution Description": "NO_RESOLUTION_DESCRIPTION" # 8.32% nulos
}

df_imputed = df_drop.na.fill(text_impute_map)



3.3 Imputando valores nulos en columnas categóricas...


In [9]:
# 3.4 Imputación/Gestión de campos Numéricos y Casting
print("\n3.4 Imputando/gestionando valores nulos en columnas numéricas y casteando tipos...")

# Cuantificar filas eliminadas por nulos en coordenadas
count_before_dropna_coords = df_imputed.count()
print(f"Filas antes de eliminar nulos en coordenadas (Lat/Lon, X/Y): {count_before_dropna_coords}")

# CRÍTICO: Eliminar filas con nulos en lat/lon y coordenadas X/Y (3.34% de los datos)
# Es mejor perder unas pocas filas que tener ubicaciones erróneas para la predicción de accidentalidad.
df_imputed = df_imputed.dropna(subset=["latitude", "longitude", "X Coordinate (State Plane)", "Y Coordinate (State Plane)"])

count_after_dropna_coords = df_imputed.count()
rows_dropped_coords = count_before_dropna_coords - count_after_dropna_coords
print(f"Filas después de eliminar nulos en coordenadas: {count_after_dropna_coords}")
print(f"  -> Filas eliminadas debido a nulos en coordenadas: {rows_dropped_coords}")
if count_before_dropna_coords > 0: # Evitar división por cero
    print(f"  -> Porcentaje de filas eliminadas por nulos en coordenadas: {(rows_dropped_coords / count_before_dropna_coords) * 100:.2f}%")

# Para Incident Zip (2.73% nulos): imputar con un valor que indique "desconocido" numéricamente
# Usamos -1 para indicar zip desconocido, ya que 0 podría ser un zip code válido y no queremos la mediana aquí.
df_imputed = df_imputed.na.fill({"Incident Zip": -1})

# Castear lat/lon a DoubleType (esto no maneja nulos, solo cambia el tipo)
df_imputed = df_imputed.withColumn("latitude", F.col("latitude").cast(DoubleType()))
df_imputed = df_imputed.withColumn("longitude", F.col("longitude").cast(DoubleType()))



3.4 Imputando/gestionando valores nulos en columnas numéricas y casteando tipos...
Filas antes de eliminar nulos en coordenadas (Lat/Lon, X/Y): 209999
Filas después de eliminar nulos en coordenadas: 202975
  -> Filas eliminadas debido a nulos en coordenadas: 7024
  -> Porcentaje de filas eliminadas por nulos en coordenadas: 3.34%


In [10]:
# 3.5 Normalización de Variables Categóricas (pasar a mayúsculas y quitar espacios)
print("\n3.5 Normalizando columnas categóricas (mayúsculas y trim)...")
cat_cols_to_normalize = [
    "Agency", "Agency Name", "Complaint Type", "Descriptor", "Location Type", "City", "Borough",
    "City Council Districts", "Police Precincts", "Community Districts", "Borough Boundaries",
    "Park Facility Name", "Open Data Channel Type", "Park Borough", "BBL", "Resolution Description",
    "Street Name", "Incident Address", "Community Board" # Agregadas para consistencia
]

for c in cat_cols_to_normalize:
    if c in df_imputed.columns:
        df_imputed = df_imputed.withColumn(c, trim(upper(col(c))))



3.5 Normalizando columnas categóricas (mayúsculas y trim)...


In [11]:

# --- 4. Verificación del DataFrame Final ---
print("\n4. Primeros 5 registros del DataFrame después de limpieza e imputación:")
df_imputed.show(5, truncate=False) # truncate=False para ver el contenido completo de las columnas



4. Primeros 5 registros del DataFrame después de limpieza e imputación:
+----------+-------------------+-------------------+------+--------------------------------+--------------------------+--------------------------------+-------------+------------+------------------+------------------+-------------+-----------+----------------------------------------------------------------------------------------------------+------------------------------+----------------+----------+-------------+--------------------------+--------------------------+----------------------+------------------+-------------+-----------------+------------------+-------------------+------------------+----------------------+----------------+
|Unique Key|Created Date       |Closed Date        |Agency|Agency Name                     |Complaint Type            |Descriptor                      |Location Type|Incident Zip|Incident Address  |Street Name       |City         |Status     |Resolution Description                  

In [12]:
# --- 5. Guardar el DataFrame Limpio en Silver como Parquet ---
print("\n6. Guardando datos limpios en Silver como Parquet...")
df_imputed.write \
  .mode("overwrite") \
  .parquet("s3://proyecto-ny311/silver/ny311/")

print("Datos guardados en Silver correctamente.")

# --- 7. Finalizar el Job de Glue ---
job.commit()
print("Job de Glue finalizado.")


6. Guardando datos limpios en Silver como Parquet...
Datos guardados en Silver correctamente.
Job de Glue finalizado.
